# 07 — Training on OpenWebText

Same GPT as notebook 06 (imported from `scripts/gpt.py`), but the data no longer fits
in memory, so the batch getter memory-maps the extracted text files.

**Prerequisite**: download + extract OpenWebText and run
`python scripts/data_extract.py --src openwebtext --out_dir data` (see
`docs/05_openwebtext.md`). That creates `data/output_train.txt`, `data/output_val.txt`
and `data/vocab.txt`. Set `DATA_DIR` below if they live elsewhere.

> **Note on the saved outputs:** the corpus isn't in this repo (~40 GB). The outputs
> below come from a smoke-test run on a tiny sample packaged in the same `.xz` archive
> format (the Wizard of Oz text), to show the pipeline works end to end. Re-run the
> notebook after extracting OpenWebText to get real results.

In [1]:
import mmap
import os
import pickle
import random
import sys
import time

import torch

sys.path.insert(0, os.path.abspath('../scripts'))
from gpt import CharTokenizer, GPTConfig, GPTLanguageModel, get_device

DATA_DIR = os.environ.get('DATA_DIR', '../data')
MODEL_PATH = 'model-01.pkl'
device = get_device()
print(device)

cpu


## Hyperparameters
Bump these up on a GPU (e.g. `batch_size=64, block_size=128, n_embd=384, n_head=8,
n_layer=8, max_iters=20000`). If you hit `CUDA out of memory`, lower `batch_size` or
`block_size` first — watch *Dedicated GPU memory* in Task Manager (Windows) or
`nvidia-smi` to see how close you are to the limit.

In [2]:
batch_size = 32
block_size = 64
max_iters = 1000
eval_interval = 250
eval_iters = 20
learning_rate = 3e-4
n_embd, n_head, n_layer, dropout = 128, 4, 4, 0.2
torch.manual_seed(1337)
random.seed(1337)

## Vocabulary from `vocab.txt`

In [3]:
with open(os.path.join(DATA_DIR, 'vocab.txt'), 'r', encoding='utf-8') as f:
    tokenizer = CharTokenizer(f.read())
vocab_size = tokenizer.vocab_size
print('vocab size:', vocab_size)

vocab size: 80


## Adjusted dataloader: random chunk via mmap → batch
Train/val splits are now two separate files (made by the extractor), so `split` picks
the file instead of slicing a tensor.

In [4]:
def get_random_chunk(split):
    filename = os.path.join(DATA_DIR, 'output_train.txt' if split == 'train' else 'output_val.txt')
    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            file_size = len(mm)
            start_pos = random.randint(0, file_size - block_size * batch_size)
            mm.seek(start_pos)
            block = mm.read(block_size * batch_size - 1)
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', '')
            data = torch.tensor(tokenizer.encode(decoded_block), dtype=torch.long)
    return data

def get_batch(split):
    data = get_random_chunk(split)
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

x, y = get_batch('train')
print(x.shape, y.shape)
print(repr(tokenizer.decode(x[0].tolist())))

torch.Size([32, 64]) torch.Size([32, 64])
"ough the Munchkin was hardly tall enough to\ncome to Zeb's should"


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Model loading / saving
If a pickled model exists we continue training it, otherwise we start fresh.

In [6]:
if os.path.exists(MODEL_PATH):
    print('loading model parameters...')
    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)['model']
    print('loaded successfully!')
else:
    config = GPTConfig(vocab_size=vocab_size, block_size=block_size, n_embd=n_embd,
                       n_head=n_head, n_layer=n_layer, dropout=dropout)
    model = GPTLanguageModel(config)
model = model.to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters')

0.82M parameters


## Training on OpenWebText

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
start = time.time()
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f} ({time.time() - start:.0f}s)")
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.449, val loss: 4.438 (1s)


step: 250, train loss: 2.435, val loss: 2.498 (29s)


step: 500, train loss: 2.254, val loss: 2.328 (57s)


step: 750, train loss: 1.994, val loss: 2.112 (85s)


step: 999, train loss: 1.824, val loss: 1.944 (112s)
1.9310964345932007


## Pickling the trained model

In [8]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump({'model': model.to('cpu'), 'tokenizer': tokenizer}, f)
model.to(device)
print('model saved')

model saved


## Prompt → completion

In [9]:
prompt = 'Hello! Can you see me?'
context = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
print(tokenizer.decode(model.generate(context, max_new_tokens=200)[0].tolist()))

Hello! Can you see me?"

He a and shoRow ent Pemesedrelaretch, with seapiesiked the hey breo tachee;
Wngabut, bein'pp id thes. The ifere intle Ucashed tors the pitably a
thorses, wilblr ther bet justy timanchich tiouw, wer
